path=/Volumes/workspace/default/pyspark_datasets/List of Orders.csv

In [0]:
df=spark.read.format("csv").option("header","true").load("/Volumes/workspace/default/pyspark_datasets/List of Orders.csv")
display(df)

Order ID,Order Date,CustomerName,State,City
B-25601,01-04-2018,Bharat,Gujarat,Ahmedabad
B-25602,01-04-2018,Pearl,Maharashtra,Pune
B-25603,03-04-2018,Jahan,Madhya Pradesh,Bhopal
B-25604,03-04-2018,Divsha,Rajasthan,Jaipur
B-25605,05-04-2018,Kasheen,West Bengal,Kolkata
B-25606,06-04-2018,Hazel,Karnataka,Bangalore
B-25607,06-04-2018,Sonakshi,Jammu and Kashmir,Kashmir
B-25608,08-04-2018,Aarushi,Tamil Nadu,Chennai
B-25609,09-04-2018,Jitesh,Uttar Pradesh,Lucknow
B-25610,09-04-2018,Yogesh,Bihar,Patna


Scenario ecommerce company

Problem in raw data is:

You have some missing values, wrong data types, duplicate values, dirty columns

In [0]:
data = [
    (1, "C001", "Laptop", "Electronics", "50000", "2", "2024-01-10"),
    (2, "C002", "Shoes", "Fashion", "2000", "1", "2024-01-11"),
    (3, "C001", "Mouse", "Electronics", "abc", "1", "2024-01-12"),
    (4, "C003", "T-shirt", "Fashion", "1500", None, "2024-01-13"),
    (5, "C004", "Mobile", "Electronics", "30000", "1", "2024-01-14"),
    (6, "C002", "Shoes", "Fashion", "2000", "1", "2024-01-11"),
]
columns = ["Order_id", "Customer_id", "Product", "Category", "Price", "Quantity", "Order_date"]
df_raw = spark.createDataFrame(data, columns)
#display(df_raw)
df_raw.show()

+--------+-----------+-------+-----------+-----+--------+----------+
|Order_id|Customer_id|Product|   Category|Price|Quantity|Order_date|
+--------+-----------+-------+-----------+-----+--------+----------+
|       1|       C001| Laptop|Electronics|50000|       2|2024-01-10|
|       2|       C002|  Shoes|    Fashion| 2000|       1|2024-01-11|
|       3|       C001|  Mouse|Electronics|  abc|       1|2024-01-12|
|       4|       C003|T-shirt|    Fashion| 1500|    NULL|2024-01-13|
|       5|       C004| Mobile|Electronics|30000|       1|2024-01-14|
|       6|       C002|  Shoes|    Fashion| 2000|       1|2024-01-11|
+--------+-----------+-------+-----------+-----+--------+----------+



In [0]:
df_raw.printSchema()

root
 |-- Order_id: long (nullable = true)
 |-- Customer_id: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Order_date: string (nullable = true)



In [0]:
#Data cleaning
from pyspark.sql.functions import expr, col

df_clean = df_raw \
    .dropDuplicates() \
    .withColumn("price", expr("try_cast(Price as int)")) \
    .withColumn("quantity", col("Quantity").cast("int")) \
    .fillna({"quantity": 1})

df_clean.show()

#display(df_clean)

#dropDuplicates removes duplicates records
#withColumn convert price into integer using cast()
#fillna helps to assume value 1 if any value is missing or null 

+--------+-----------+-------+-----------+-----+--------+----------+
|Order_id|Customer_id|Product|   Category|price|quantity|Order_date|
+--------+-----------+-------+-----------+-----+--------+----------+
|       1|       C001| Laptop|Electronics|50000|       2|2024-01-10|
|       2|       C002|  Shoes|    Fashion| 2000|       1|2024-01-11|
|       3|       C001|  Mouse|Electronics| NULL|       1|2024-01-12|
|       4|       C003|T-shirt|    Fashion| 1500|       1|2024-01-13|
|       5|       C004| Mobile|Electronics|30000|       1|2024-01-14|
|       6|       C002|  Shoes|    Fashion| 2000|       1|2024-01-11|
+--------+-----------+-------+-----------+-----+--------+----------+



In [0]:
df_clean.printSchema()

root
 |-- Order_id: long (nullable = true)
 |-- Customer_id: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- quantity: integer (nullable = false)
 |-- Order_date: string (nullable = true)



In [0]:
#Scenario to create  business column
df_clean=df_clean.withColumn(
    "total_amount", col("price")*col("quantity")
)
df_clean.show()


+--------+-----------+-------+-----------+-----+--------+----------+------------+
|Order_id|Customer_id|Product|   Category|price|quantity|Order_date|total_amount|
+--------+-----------+-------+-----------+-----+--------+----------+------------+
|       1|       C001| Laptop|Electronics|50000|       2|2024-01-10|      100000|
|       2|       C002|  Shoes|    Fashion| 2000|       1|2024-01-11|        2000|
|       3|       C001|  Mouse|Electronics| NULL|       1|2024-01-12|        NULL|
|       4|       C003|T-shirt|    Fashion| 1500|       1|2024-01-13|        1500|
|       5|       C004| Mobile|Electronics|30000|       1|2024-01-14|       30000|
|       6|       C002|  Shoes|    Fashion| 2000|       1|2024-01-11|        2000|
+--------+-----------+-------+-----------+-----+--------+----------+------------+



In [0]:
from pyspark.sql.functions import col

df_clean = df_clean.withColumn(
    "total_amount",
    col("price") * col("quantity")
)

In [0]:
#select Expression
df_clean.selectExpr(
    "Order_id",
    "Customer_id",
    "price",
    "quantity",
    "total_amount"
).show()


+--------+-----------+-----+--------+------------+
|Order_id|Customer_id|price|quantity|total_amount|
+--------+-----------+-----+--------+------------+
|       1|       C001|50000|       2|      100000|
|       2|       C002| 2000|       1|        2000|
|       3|       C001| NULL|       1|        NULL|
|       4|       C003| 1500|       1|        1500|
|       5|       C004|30000|       1|       30000|
|       6|       C002| 2000|       1|        2000|
+--------+-----------+-----+--------+------------+



In [0]:
df_clean.filter(col("price").isNull()).show()

+--------+-----------+-------+-----------+-----+--------+----------+------------+
|Order_id|Customer_id|Product|   Category|price|quantity|Order_date|total_amount|
+--------+-----------+-------+-----------+-----+--------+----------+------------+
|       3|       C001|  Mouse|Electronics| NULL|       1|2024-01-12|        NULL|
+--------+-----------+-------+-----------+-----+--------+----------+------------+

